# Python `collections` Module

The `collections` module provides specialised container types that extend the built-in `dict`, `list`, `set`, and `tuple`.

| Type | One-line description |
|---|---|
| `namedtuple` | Tuple with named fields — readable, immutable, memory-efficient |
| `deque` | Double-ended queue — O(1) append/pop from both ends |
| `Counter` | Dict for counting hashable items |
| `defaultdict` | Dict that auto-creates missing values via a factory |
| `OrderedDict` | Dict that preserves insertion order with extra ordering operations |
| `ChainMap` | Single view over multiple dicts — lookups search each in order |

## `namedtuple`

A factory that creates a `tuple` subclass with **named fields**.  
Behaves exactly like a plain tuple (immutable, indexable, unpackable) but fields are also accessible by name, making code far more readable than numeric indexing.

In [1]:
from collections import namedtuple

Point = namedtuple('Point', ['x', 'y'])
Card  = namedtuple('Card',  ['rank', 'suit'])

p = Point(3, 4)
c = Card('Ace', 'Spades')

# named field access
print(p.x, p.y)

# still a tuple — index and unpack work
x, y = p
print(x, y)
print(p[0])

# _replace returns a new instance with changed fields
p2 = p._replace(y=10)
print(p2)

# _asdict is handy for serialisation
print(c._asdict())

3 4
3 4
3
Point(x=3, y=10)
{'rank': 'Ace', 'suit': 'Spades'}


## `deque`

A **double-ended queue** backed by a doubly-linked list.  
Appending or popping from either end is **O(1)**, whereas `list.pop(0)` (left-end removal) is O(n) because every remaining element must shift.  
The optional `maxlen` parameter makes it a fixed-size sliding window — old items are evicted automatically.

In [2]:
from collections import deque

dq = deque([1, 2, 3])

# O(1) operations on both ends
dq.appendleft(0)
dq.append(4)
print(dq)

dq.popleft()
dq.pop()
print(dq)

# rotate shifts elements right (positive) or left (negative)
dq.rotate(1)
print(dq)

# maxlen — last N items kept automatically
recent = deque(maxlen=3)
for n in range(6):
    recent.append(n)
    print(f"  added {n}: {recent}")

deque([0, 1, 2, 3, 4])
deque([1, 2, 3])
deque([3, 1, 2])
  added 0: deque([0], maxlen=3)
  added 1: deque([0, 1], maxlen=3)
  added 2: deque([0, 1, 2], maxlen=3)
  added 3: deque([1, 2, 3], maxlen=3)
  added 4: deque([2, 3, 4], maxlen=3)
  added 5: deque([3, 4, 5], maxlen=3)


## `Counter`

A `dict` subclass for **counting hashable objects**.  
Missing keys return `0` instead of raising `KeyError`.  
Supports `most_common()`, arithmetic between counters (`+`, `-`, `&`, `|`), and can be initialised from any iterable.

In [3]:
from collections import Counter

words = ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']
c = Counter(words)

print(c)

# top N most common
print(c.most_common(2))

# missing key returns 0, not KeyError
print(c['mango'])

# counter arithmetic — add two counters
more = Counter({'banana': 2, 'date': 5})
print(c + more)

# count characters in a string directly
letter_freq = Counter('mississippi')
print(letter_freq.most_common(3))

Counter({'apple': 3, 'banana': 2, 'cherry': 1})
[('apple', 3), ('banana', 2)]
0
Counter({'date': 5, 'banana': 4, 'apple': 3, 'cherry': 1})
[('i', 4), ('s', 4), ('p', 2)]


## `defaultdict`

A `dict` subclass that calls a **default factory** to create a value whenever a missing key is accessed.  
Eliminates `if key not in d` boilerplate for grouping and accumulation patterns.  
See `py_007_default_dictionary` for a full deep-dive.

In [4]:
from collections import defaultdict

# group words by first letter
grouped = defaultdict(list)
for word in ['ant', 'apple', 'bat', 'bear', 'cat']:
    grouped[word[0]].append(word)

print(dict(grouped))

# count with int factory (0 is the default)
freq = defaultdict(int)
for ch in 'abracadabra':
    freq[ch] += 1

print(dict(freq))

{'a': ['ant', 'apple'], 'b': ['bat', 'bear'], 'c': ['cat']}
{'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1}


## `OrderedDict`

A `dict` subclass that **guarantees insertion order** and adds two extra operations plain `dict` lacks:  
- `move_to_end(key, last=True/False)` — reposition a key to the front or back  
- Equality considers order: two `OrderedDict`s with the same pairs but different order are **not equal**

> Note: since Python 3.7 plain `dict` also preserves insertion order, so `OrderedDict` is mainly useful when you need `move_to_end`, order-aware equality, or an explicit signal that order matters.

In [5]:
from collections import OrderedDict

od = OrderedDict()
od['first']  = 1
od['second'] = 2
od['third']  = 3

print(list(od.keys()))

# move 'first' to the end
od.move_to_end('first')
print(list(od.keys()))

# move 'first' back to the front
od.move_to_end('first', last=False)
print(list(od.keys()))

# order-aware equality — plain dict ignores order
od1 = OrderedDict([('a', 1), ('b', 2)])
od2 = OrderedDict([('b', 2), ('a', 1)])
print(f"OrderedDict equal: {od1 == od2}")
print(f"plain dict equal : {dict(od1) == dict(od2)}")

['first', 'second', 'third']
['second', 'third', 'first']
['first', 'second', 'third']
OrderedDict equal: False
plain dict equal : True


## `ChainMap`

Groups multiple dicts (or mappings) into a **single view** without copying any data.  
Lookups search each map left-to-right and return the first hit.  
Writes always go to the **first** map.  
Classic use-case: layered config (local overrides → env overrides → defaults).

In [6]:
from collections import ChainMap

defaults = {'color': 'blue',  'font': 'Arial', 'size': 12}
env      = {'color': 'green', 'size': 14}
local    = {'color': 'red'}

# local wins over env, env wins over defaults
config = ChainMap(local, env, defaults)

print(config['color'])
print(config['font'])
print(config['size'])

# writes go to the first map only
config['size'] = 20
print(f"local after write: {local}")
print(f"env unchanged    : {env}")

# new_child() pushes a fresh empty map to the front
child = config.new_child({'font': 'Courier'})
print(f"child font: {child['font']}, parent font: {config['font']}")

red
Arial
14
local after write: {'color': 'red', 'size': 20}
env unchanged    : {'color': 'green', 'size': 14}
child font: Courier, parent font: Arial
